# Identify analyses to rerun

Requested combinations (`get_analyses_for_country`) that are not skipped (no scaler data) and have no usable folder in `FULL_RUN`. Usable is defined as finished outputs, a non-tiny idata, and current methods for OxCGRT and Oceania/Singapore. Jobs are searched in list order, with the first usable copy found in a given job used for that country-analysis type. The remainder are written out to `data/config/rerun_pairs.json`.


In [ ]:
import json
import pandas as pd

from emu_renewal.constants import ANALYSIS_TYPES, DATA_PATH, OUTPUTS_PATH, FULL_RUN
from emu_renewal.run import get_analyses_for_country
from emu_renewal.utils import is_usable_analysis

In [ ]:
job_ids = FULL_RUN
countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))

In [ ]:
def classify_analysis(iso3, analysis, run_path, log_text):
    if log_text and f"{analysis} data not available" in log_text:
        return "skipped"
    if is_usable_analysis(run_path / iso3 / analysis, iso3, analysis):
        return "usable"
    return "not usable"


def classify_job(run_id):
    run_path = OUTPUTS_PATH / run_id
    status = pd.DataFrame(index=countries, columns=ANALYSIS_TYPES)
    for iso3 in countries:
        log_path = run_path / iso3 / "run.log"
        log_text = log_path.read_text() if log_path.exists() else None
        requested_types = get_analyses_for_country(iso3)
        for analysis in ANALYSIS_TYPES:
            if analysis not in requested_types:
                status.loc[iso3, analysis] = "not requested"
            else:
                status.loc[iso3, analysis] = classify_analysis(iso3, analysis, run_path, log_text)
    return status


statuses = {job: classify_job(job) for job in job_ids}
pd.concat(
    {job: s.apply(pd.Series.value_counts).fillna(0).astype(int) for job, s in statuses.items()},
    axis=1,
).fillna(0).astype(int)

In [ ]:
def pairs_from_mask(mask):
    stacked = mask.stack()
    return list(stacked[stacked].index)


requested = pd.DataFrame(False, index=countries, columns=ANALYSIS_TYPES)
for iso3 in countries:
    requested.loc[iso3, get_analyses_for_country(iso3)] = True

skipped = pd.DataFrame(False, index=countries, columns=ANALYSIS_TYPES)
claimed = pd.DataFrame(False, index=countries, columns=ANALYSIS_TYPES)
summary = {"requested": int(requested.to_numpy().sum())}
for job in job_ids:
    skipped = skipped | (statuses[job] == "skipped")
    usable = statuses[job] == "usable"
    summary[f"available from {job}"] = int((requested & usable & ~claimed).to_numpy().sum())
    claimed = claimed | usable

summary["skipped"] = int((requested & skipped).to_numpy().sum())
need = requested & ~skipped & ~claimed
rerun_pairs = sorted(pairs_from_mask(need))
summary["remaining"] = len(rerun_pairs)
pd.Series(summary)

In [ ]:
pd.Series([analysis for _, analysis in rerun_pairs]).value_counts()

In [ ]:
json.dump(rerun_pairs, open(DATA_PATH / "config/rerun_pairs.json", "w"))